In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW all_dx_claims AS

-- Medical Events - Dx (COALESCE)
SELECT DISTINCT 
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'DX' AS CLAIM_TYPE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
  AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-11-30'

UNION

-- Pharmacy Events - Dx (PRESCRIBER_NPI)
SELECT DISTINCT 
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    FILL_DATE,
    'DX' AS CLAIM_TYPE
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE DIAGNOSIS_CODE IN ('E761', 'E763')
  AND TRANSACTION_STATUS = 'PAID'
  AND FILL_DATE BETWEEN '2020-08-01' AND '2025-11-30';


-- =============================================================================
-- STEP 2: TREATMENT CLAIMS (5yr) - CORRECTED NPI LOGIC
-- =============================================================================

CREATE OR REPLACE TEMPORARY VIEW all_tx_claims AS

-- Medical Events - NDC codes (COALESCE)
SELECT DISTINCT 
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'TX' AS CLAIM_TYPE,
    NDC11 AS CODE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE NDC11 IN ('54092070001', '540920700')
  AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-11-30'

UNION

-- Medical Events - Procedure codes (RENDERING_NPI only)
SELECT DISTINCT 
    PATIENT_ID,
    RENDERING_NPI AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'TX' AS CLAIM_TYPE,
    PROCEDURE_CODE AS CODE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE PROCEDURE_CODE IN ('99601', '99602', '96365', '96366', 'J1743', 
                         'S9357', 'S9379', '38206', '38230', '38232', 
                         '38240', '38241', '38242', '38243', '38250')
  AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-11-30'

UNION

-- Pharmacy Events - NDC codes (PRESCRIBER_NPI)
SELECT DISTINCT 
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    FILL_DATE,
    'TX' AS CLAIM_TYPE,
    NDC11 AS CODE
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE NDC11 IN ('54092070001', '540920700')
  AND TRANSACTION_RESULT = 'PAID'
  AND FILL_DATE BETWEEN '2020-08-01' AND '2025-11-30';


-- =============================================================================
-- STEP 3: TREATMENT CLAIMS FOR ELIGIBILITY (2yr)
-- =============================================================================

CREATE OR REPLACE TEMPORARY VIEW tx_claims_2yr AS
SELECT DISTINCT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE, CODE
FROM all_tx_claims
WHERE FILL_DATE BETWEEN '2023-08-01' AND '2025-11-30';


-- =============================================================================
-- STEP 4: PATIENT ELIGIBILITY
-- =============================================================================

-- Specified: 2+ E761 Dx dates (5yr)
CREATE OR REPLACE TEMPORARY VIEW e761_patients_2dx AS
SELECT PATIENT_ID
FROM (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-11-30'
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '2025-11-30'
)
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 2;


-- Specified Patients: 2+ E761 Dx + any Tx in 2yr
CREATE OR REPLACE TEMPORARY VIEW specified_patients AS
SELECT DISTINCT e.PATIENT_ID
FROM e761_patients_2dx e
INNER JOIN tx_claims_2yr t ON e.PATIENT_ID = t.PATIENT_ID;


-- Incremental: 2+ E763 Dx dates (5yr)
CREATE OR REPLACE TEMPORARY VIEW e763_patients_2dx AS
SELECT PATIENT_ID
FROM (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-11-30'
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '2025-11-30'
)
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 2;


-- Elaprase Tx in 2yr (for incremental eligibility)
CREATE OR REPLACE TEMPORARY VIEW elaprase_tx_2yr AS
SELECT DISTINCT PATIENT_ID
FROM tx_claims_2yr
WHERE CODE IN ('54092070001', '540920700', 'J1743');


-- Incremental Patients: 2+ E763 Dx + Elaprase Tx in 2yr + NOT specified
CREATE OR REPLACE TEMPORARY VIEW incremental_patients AS
SELECT DISTINCT e.PATIENT_ID
FROM e763_patients_2dx e
INNER JOIN elaprase_tx_2yr t ON e.PATIENT_ID = t.PATIENT_ID
WHERE e.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM specified_patients);


-- All Eligible Patients
CREATE OR REPLACE TEMPORARY VIEW eligible_patients AS
SELECT PATIENT_ID FROM specified_patients
UNION
SELECT PATIENT_ID FROM incremental_patients;


-- =============================================================================
-- STEP 5: COMBINED Dx + Tx CLAIMS FOR ELIGIBLE PATIENTS
-- =============================================================================

CREATE OR REPLACE TEMPORARY VIEW all_patient_claims AS

-- Dx claims (5yr)
SELECT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE
FROM all_dx_claims
WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

UNION

-- Tx claims (5yr)
SELECT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE
FROM all_tx_claims
WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients);

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW all_patients_non_unique_hcp AS

WITH hcp_level AS (
    SELECT *
    FROM all_patient_claims
)

SELECT
    TRY_CAST(npi AS STRING) AS hcp_npi,
    COUNT(DISTINCT patient_id) AS all_patients_non_unique
FROM hcp_level
WHERE npi IS NOT NULL
GROUP BY 1
ORDER BY 2 DESC;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW all_patients_unique_hcp AS

WITH hcp_metrics AS (
    SELECT 
        a.PATIENT_ID,
        a.NPI,

        CASE 
            WHEN p.primary_specialty LIKE '%Genetic%' 
              OR p.secondary_specialty LIKE '%Genetic%' 
                THEN 'Geneticist'

            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%' 
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%' 
              OR p.primary_specialty LIKE '%Neurological Surgery%' 
                THEN 'Psychiatry & Neurology'

            WHEN p.primary_specialty LIKE '%Pediatrics%' 
                THEN 'Pediatrician'

            WHEN p.primary_specialty LIKE '%Internal Medicine%' 
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%' 
              OR p.secondary_specialty LIKE '%Family Medicine%' 
                THEN 'PCP'

            WHEN p.primary_specialty LIKE '%Nurse Practitioner%' 
              OR p.primary_specialty LIKE '%Physician Assistant%' 
                THEN 'NPPA'

            WHEN a.NPI IS NULL 
                THEN 'NA'

            ELSE 'Others'
        END AS SPECIALTY,

        CASE 
            WHEN p.primary_specialty LIKE '%Genetic%' 
              OR p.secondary_specialty LIKE '%Genetic%' 
                THEN 1
            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%' 
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%' 
              OR p.primary_specialty LIKE '%Neurological Surgery%' 
                THEN 2
            WHEN p.primary_specialty LIKE '%Pediatrics%' 
                THEN 3
            WHEN p.primary_specialty LIKE '%Internal Medicine%' 
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%' 
              OR p.secondary_specialty LIKE '%Family Medicine%' 
                THEN 4
            WHEN p.primary_specialty LIKE '%Nurse Practitioner%' 
              OR p.primary_specialty LIKE '%Physician Assistant%' 
                THEN 5
            WHEN a.NPI IS NULL 
                THEN 7
            ELSE 6
        END AS SPECIALTY_PRIORITY,

        COUNT(DISTINCT a.FILL_DATE) AS NO_OF_VISITS,

        COUNT(DISTINCT CASE 
            WHEN a.CLAIM_TYPE = 'DX' THEN a.FILL_DATE 
        END) AS DX_VISITS,

        COUNT(DISTINCT CASE 
            WHEN a.CLAIM_TYPE = 'TX' THEN a.FILL_DATE 
        END) AS TX_VISITS,

        MAX(a.FILL_DATE) AS MOST_RECENT_VISIT

    FROM all_patient_claims a
    LEFT JOIN com_edp_prd.com_raw.kom_providers p 
        ON a.NPI = p.NPI

    GROUP BY 
        a.PATIENT_ID, 
        a.NPI,
        p.primary_specialty, 
        p.secondary_specialty
),

ranked_hcps AS (
    SELECT 
        *,
        RANK() OVER (
            PARTITION BY PATIENT_ID 
            ORDER BY 
                SPECIALTY_PRIORITY ASC,
                NO_OF_VISITS DESC,
                MOST_RECENT_VISIT DESC,
                NPI ASC
        ) AS HCP_RANK
    FROM hcp_metrics
),

hcp_level AS (
    SELECT 
        PATIENT_ID,
        NPI AS PRIMARY_HCP_NPI,
        SPECIALTY AS PRIMARY_HCP_SPECIALTY,
        SPECIALTY_PRIORITY,
        NO_OF_VISITS,
        DX_VISITS,
        TX_VISITS,
        MOST_RECENT_VISIT
    FROM ranked_hcps
    WHERE HCP_RANK = 1
)

-- ============================================================
-- Final UNIQUE patient count by HCP
-- Each patient contributes to exactly ONE primary HCP
-- ============================================================
SELECT
    TRY_CAST(primary_hcp_npi AS STRING) AS hcp_npi,
    primary_hcp_specialty,
    COUNT(DISTINCT patient_id) AS all_patients_unique
FROM hcp_level
WHERE primary_hcp_npi IS NOT NULL
GROUP BY 1, 2
ORDER BY 3 DESC;

In [0]:
%sql
WITH input_hcps AS (
  SELECT col1 AS hcp_npi
  FROM VALUES
('1295888642'),
('1609881366'),
('1588984942'),
('1447357405'),
('1841267051'),
('1235197864'),
('1225027782'),
('1922138437'),
('1982845327'),
('1952341885'),
('1740350024'),
('1215958756'),
('1043234461'),
('1417272188'),
('1144269341'),
('1063615672'),
('1689680613'),
('1447576434'),
('1881885531'),
('1699069583'),
('1487808275'),
('1235450693'),
('1023215217'),
('1386103216'),
('1649297573'),
('1932119377'),
('1003849134'),
('1528395985'),
('1477998730'),
('1568538858'),
('1154335719'),
('1073590550'),
('1538723002'),
('1659343341'),
('1487677340'),
('1407113178'),
('1346230398'),
('1841367604'),
('1851703284'),
('1417930678'),
('1275535395'),
('1093001448'),
('1982993390'),
('1174525711'),
('1831116656'),
('1659387272'),
('1215314265'),
('1841270451'),
('1780970020'),
('1477904068'),
('1790238921'),
('1013087998'),
('1194986430'),
('1699830547'),
('1881032050'),
('1073717948'),
('1942444617'),
('1376610139'),
('1750373353'),
('1215044557'),
('1598185126'),
('1447678230'),
('1629056536'),
('1740691427'),
('1982687422'),
('1568692812'),
('1194785196'),
('1750780011'),
('1811009939'),
('1679867808'),
('1346528072'),
('1487996203'),
('1669762746'),
('1003059932'),
('1316104508'),
('1447259494'),
('1740291848'),
('1407946957'),
('1568412005'),
('1427347962'),
('1881217453'),
('1780025585'),
('1649405499'),
('1164623682'),
('1760471569'),
('1881623239'),
('1083709471'),
('1457592479'),
('1396156931'),
('1659535839'),
('1043200173'),
('1629336276'),
('1871556878'),
('1376520056'),
('1477643856'),
('1851787626'),
('1952795403'),
('1154416675'),
('1427685916'),
('1609040716'),
('1447287933'),
('1750675161'),
('1114964228'),
('1699850057'),
('1174884225'),
('1710302880'),
('1871629246'),
('1174524011'),
('1336170059'),
('1023379336'),
('1861753303'),
('1679597983'),
('1578055935'),
('1588989784'),
('1043331044'),
('1588714463'),
('1174713168'),
('1669188645'),
('1104888098'),
('1407066384'),
('1609408210'),
('1912086760'),
('1679356372'),
('1235949199'),
('1730322462'),
('1548645237'),
('1073909933'),
('1144286980'),
('1194213678'),
('1265922702'),
('1346526324'),
('1043491483'),
('1760547418'),
('1114148020'),
('1255416012'),
('1215989132'),
('1558387308'),
('1679586275'),
('1609887983'),
('1497011225'),
('1801082896'),
('1922458736'),
('1972861441'),
('1588005870'),
('1447270053'),
('1467896464'),
('1548666258'),
('1538175286'),
('1184687162'),
('1710171715'),
('1487734091'),
('1366093023'),
('1710065271'),
('1265463772'),
('1851313696'),
('1104138676'),
('1023404597'),
('1093714040'),
('1669976098'),
('1881649796'),
('1477146298'),
('1619065190'),
('1902969546'),
('1992060958'),
('1144261595'),
('1558339358'),
('1831157916'),
('1508413147'),
('1497969786'),
('1063153633'),
('1619635471'),
('1659094928'),
('1861673972'),
('1962629600'),
('1699718031'),
('1770891020'),
('1134443617'),
('1184642951'),
('1235150491'),
('1255758637'),
('1568469245'),
('1205958964'),
('1316151814'),
('1942460837'),
('1215961347'),
('1508182049'),
('1306016860'),
('1336572544'),
('1144715251'),
('1346212966'),
('1881907533'),
('1568562445'),
('1538396007'),
('1265405120'),
('1245471234'),
('1932177151'),
('1902260722'),
('1932117470'),
('1548266299'),
('1780817825'),
('1295930212'),
('1376575175'),
('1891085528'),
('1245625078'),
('1912085788'),
('1124187018'),
('1174525059'),
('1255400891'),
('1760464044'),
('1982281432'),
('1447297551'),
('1093039034'),
('1235514688'),
('1639339450'),
('1770719841'),
('1922051762'),
('1528038114'),
('1578939864'),
('1346482361'),
('1124694468'),
('1275882599'),
('1225281470'),
('1235623018'),
('1942628912'),
('1275964504'),
('1366083040'),
('1184656613'),
('1194878512'),
('1952620791'),
('1669453148'),
('1881230811'),
('1477865731'),
('1912936725'),
('1851785836'),
('1598955973'),
('1841456555'),
('1306525597'),
('1689002628'),
('1730508649'),
('1932573912'),
('1043240500'),
('1689988446'),
('1376705335'),
('1821000159'),
('1962460741'),
('1669456133'),
('1255576542'),
('1972000529'),
('1215939863'),
('1528267523'),
('1932337144'),
('1952475279'),
('1144253154'),
('1598928103'),
('1700267978'),
('1952313298'),
('1336298405'),
('1982661997'),
('1265694244'),
('1144375965'),
('1164695730'),
('1407995400'),
('1578019238'),
('1124224936'),
('1619302338'),
('1164408274'),
('1780656173'),
('1346465630'),
('1770711269'),
('1629178306'),
('1649776618'),
('1518420645'),
('1366490492'),
('1891871877'),
('1518059369'),
('1770639361'),
('1477537405'),
('1346512167'),
('1245317189'),
('1851393474'),
('1720620198'),
('1083690572'),
('1710242730'),
('1699933903'),
('1780624577'),
('1003858036'),
('1902403843'),
('1083929269'),
('1407464233'),
('1154835593'),
('1053532366'),
('1225026867'),
('1174753529'),
('1598376949'),
('1104935840'),
('1710961602'),
('1336149392'),
('1770564825'),
('1922271188'),
('1669459798'),
('1598779811'),
('1154465672'),
('1083946685'),
('1770678930'),
('1164989497'),
('1912258302'),
('1114136934'),
('1205213469'),
('1245438167'),
('1336109917'),
('1427311927'),
('1710203518'),
('1730457789'),
('1326079013'),
('1639322407'),
('1104389394'),
('1275792178'),
('1154639623'),
('1245410844'),
('1639221021'),
('1134516982'),
('1912226721'),
('1427264969'),
('1649765322'),
('1851435408'),
('1326083312'),
('1427391705'),
('1326529132'),
('1730250796'),
('1326492851'),
('1548289093'),
('1528355997'),
('1841393394'),
('1144217886'),
('1871976639'),
('1841480217'),
('1033489794'),
('1518192616'),
('1578542130'),
('1710440086'),
('1972029544'),
('1376868158'),
('1518634427'),
('1992899413'),
('1457951485'),
('1487128989'),
('1710446190'),
('1316520422'),
('1831545326'),
('1649415837'),
('1376769943'),
('1245226059'),
('1467048520'),
('1902446081'),
('1720207996'),
('1477566222'),
('1609890334'),
('1841233798'),
('1093896326'),
('1396781696'),
('1457675415'),
('1538629894'),
('1396476685'),
('1750481800'),
('1376745018'),
('1982835518'),
('1598090326'),
('1932865177'),
('1457910424'),
('1871927111'),
('1124374137'),
('1679523435'),
('1235211848'),
('1023246600'),
('1255112389'),
('1053516583'),
('1154421642'),
('1003067331'),
('1467447748'),
('1316144645'),
('1376522912'),
('1831658855'),
('1346835162'),
('1386802387'),
('1437322310'),
('1588143887'),
('1609074301'),
('1669694857'),
('1275778136'),
('1972759629'),
('1477755155'),
('1821297490'),
('1841267267'),
('1831283837'),
('1548406010'),
('1790709319'),
('1467125260'),
('1194912592'),
('1972958122'),
('1235332701'),
('1316152127'),
('1699965186'),
('1861107716'),
('1376572107'),
('1821178237'),
('1043273766'),
('1740508928'),
('1093391260'),
('1023196896'),
('1295397933'),
('1124288907'),
('1922011030'),
('1639288723'),
('1548234784'),
('1841243755'),
('1811901424'),
('1609058338'),
('1619929080'),
('1215164074'),
('1083909733'),
('1518018076'),
('1902901242'),
('1003121799'),
('1003251943'),
('1235196106'),
('1689635047'),
('1629049853'),
('1467553131'),
('1750792792'),
('1174768030'),
('1184045148'),
('1174593735'),
('1386656767'),
('1629369798'),
('1780146266'),
('1205938156'),
('1568429199'),
('1659472314'),
('1982044012'),
('1346332780'),
('1740442268'),
('1992810972'),
('1619406824'),
('1518131838'),
('1013987262'),
('1033742739'),
('1407891963'),
('1134170939'),
('1144386087'),
('1013013424'),
('1295797330'),
('1346329810'),
('1538461728'),
('1689052300'),
('1023105301'),
('1942240080'),
('1497491021'),
('1952035172'),
('1255075032'),
('1700142155'),
('1669714275'),
('1942594080'),
('1376709519'),
('1699781609'),
('1205061389'),
('1720073992'),
('1992098503'),
('1265597124'),
('1639175649'),
('1942364690'),
('1760519367'),
('1982927976'),
('1033479720'),
('1346653284'),
('1861036139'),
('1922272954'),
('1225557762'),
('1235870247'),
('1306454582'),
('1619353703'),
('1154705606'),
('1215227293'),
('1699847426'),
('1902361595'),
('1750360764'),
('1982807053'),
('1558750596'),
('1740770809'),
('1841365822'),
('1205345741'),
('1437173085'),
('1558762203'),
('1669614236'),
('1366768640'),
('1922086404'),
('1386616993'),
('1467192310'),
('1558334862'),
('1295054542'),
('1396710588'),
('1457329773'),
('1881122018'),
('1033108980'),
('1467452276'),
('1699295493'),
('1417047390'),
('1568455533'),
('1962467878'),
('1770873978'),
('1386051464'),
('1457302879'),
('1801818653'),
('1407836299'),
('1467480814'),
('1871578666'),
('1659465987'),
('1033463054'),
('1932357217'),
('1598175127'),
('1881734283'),
('1629495023'),
('1831156728'),
('1841679198'),
('1124029756'),
('1730157686'),
('1750584199'),
('1780826271'),
('1730613811'),
('1164015947'),
('1568463537'),
('1437449311'),
('1124221981'),
('1578656658'),
('1942236492'),
('1265499214'),
('1346407558'),
('1710019369'),
('1821061060'),
('1437477411'),
('1720582422'),
('1992743546'),
('1023487261'),
('1528019684'),
('1508451188'),
('1841367752'),
('1861629321'),
('1699758433'),
('1184826273'),
('1659515567'),
('1720735913'),
('1912190547'),
('1023006798'),
('1073615928'),
('1134176183'),
('1942308192'),
('1841213972'),
('1154841617'),
('1467413658'),
('1639114424'),
('1407822539'),
('1487066791'),
('1376628156'),
('1295824787'),
('1396037750'),
('1013996214'),
('1336618289'),
('1376511840'),
('1952445900'),
('1134337512'),
('1619985462'),
('1407417520'),
('1033538699'),
('1205421633'),
('1114019080'),
('1346216652'),
('1821258583'),
('1912114034'),
('1265415913'),
('1871555805'),
('1225352297'),
('1326206335'),
('1548239577'),
('1609882703'),
('1073600177'),
('1619997830'),
('1184088676'),
('1750677894'),
('1790103067'),
('1851553085'),
('1598379919'),
('1154413037'),
('1487090601'),
('1861811242'),
('1003831512'),
('1134447444'),
('1235695818'),
('1558685578'),
('1831113919'),
('1366774119'),
('1740495753'),
('1902972284'),
('1467023176'),
('1487752655'),
('1255773966'),
('1578616207'),
('1720355928'),
('1093898447'),
('1447273362'),
('1548478977'),
('1629020482'),
('1366888984'),
('1053418145'),
('1144580424'),
('1407833353'),
('1053907709'),
('1184159139'),
('1679552392'),
('1174545644'),
('1538166129'),
('1558311431'),
('1891307302'),
('1750024089'),
('1134129695'),
('1972389831'),
('1841884079'),
('1689692931'),
('1043426539'),
('1538795406'),
('1679777528'),
('1689616997'),
('1013972231'),
('1437123619'),
('1467688523'),
('1891966560'),
('1538320916'),
('1821418674'),
('1710011390'),
('1841238011'),
('1891421277'),
('1215407093'),
('1912401670'),
('1831385053'),
('1396748950'),
('1457462426'),
('1194373498'),
('1275583494'),
('1669960225'),
('1730371030'),
('1346799376'),
('1437214608'),
('1942533443'),
('1174916472'),
('1275709644'),
('1245348374'),
('1508386947'),
('1295889905'),
('1811969595'),
('1003084377'),
('1023176880'),
('1255581070'),
('1699209155'),
('1245506914'),
('1598722233'),
('1609910033'),
('1558654392'),
('1689688681'),
('1972560464'),
('1689328551'),
('1831292036'),
('1265410708'),
('1235405986'),
('1255716320'),
('1487038139'),
('1336137009'),
('1114390663'),
('1700157708'),
('1114901337'),
('1265423941'),
('1184702409'),
('1124447313'),
('1568719292'),
('1316113905'),
('1437159365'),
('1740443332'),
('1306360540'),
('1447519921'),
('1326138983'),
('1629249859'),
('1164482386'),
('1053349159'),
('1487602728'),
('1346237195'),
('1386645125'),
('1669538815'),
('1578970349'),
('1659517977'),
('1245641620'),
('1790896215'),
('1124380258'),
('1598702094'),
('1053483792'),
('1992880215'),
('1518928456'),
('1639597396'),
('1659540375'),
('1669687596'),
('1962675264'),
('1649372079'),
('1992860621'),
('1992142624'),
('1295052637'),
('1164800264'),
('1306130786'),
('1891780342'),
('1740440916'),
('1093041808'),
('1336376318'),
('1922273580'),
('1366492779'),
('1114997350'),
('1538598701'),
('1235346198'),
('1952394520'),
('1497744817'),
('1114001856'),
('1730295478'),
('1942864574'),
('1376833848'),
('1376793331'),
('1518227305'),
('1831360304'),
('1063065506'),
('1659931996'),
('1982709564'),
('1114942265'),
('1750591004'),
('1144220088'),
('1902122575'),
('1275598047'),
('1588771141'),
('1720382054'),
('1154387801'),
('1376592865'),
('1932424231'),
('1801207543'),
('1003201658'),
('1164429114'),
('1629090022'),
('1386176303'),
('1285807990'),
('1831391218'),
('1922073451'),
('1871597815'),
('1891988598'),
('1104970359'),
('1134232598'),
('1912966888'),
('1356575088'),
('1013952142'),
('1811309248'),
('1366489650'),
('1194106534'),
('1902217748'),
('1053673392'),
('1366407926'),
('1437131323'),
('1154392462'),
('1356792121'),
('1538239553'),
('1831405166'),
('1386646735'),
('1891784617'),
('1730159906'),
('1790042851'),
('1760503205'),
('1679525125'),
('1891708368'),
('1104084516'),
('1407975469'),
('1154620417'),
('1457763583'),
('1023044856'),
('1174828065'),
('1699754994'),
('1699033860'),
('1083843577'),
('1629335849'),
('1871560409'),
('1619921442'),
('1417158130'),
('1578876249'),
('1689619033'),
('1033143441'),
('1013918341'),
('1760860753'),
('1306100623'),
('1740227388'),
('1073999843'),
('1922119957'),
('1932527116'),
('1215298906'),
('1801231709'),
('1710281209'),
('1174526263'),
('1154580173'),
('1801983796'),
('1912260985'),
('1457677981'),
('1730289240'),
('1487944799'),
('1760748149'),
('1770894438'),
('1427112416'),
('1588679039'),
('1356306427'),
('1669892642'),
('1891774345'),
('1174504096'),
('1851475271'),
('1164661922'),
('1366468241'),
('1346205721'),
('1417333139'),
('1770565798'),
('1619980547'),
('1992790760'),
('1467849356'),
('1558470310'),
('1659342889'),
('1215039227'),
('1871564682'),
('1124059597'),
('1467689349'),
('1265550677'),
('1467417410'),
('1457593667'),
('1528170685'),
('1871564443'),
('1942578208'),
('1427410885'),
('1083642375'),
('1174562417'),
('1588680094'),
('1992188601'),
('1568419471'),
('1730274796'),
('1285645143'),
('1619231560'),
('1962498394'),
('1013296730'),
('1417968157'),
('1730419086'),
('1386809150'),
('1548325707'),
('1609131770'),
('1912106220'),
('1912105826'),
('1134153422'),
('1790750404'),
('1952425118'),
('1952564015'),
('1700815271'),
('1942460290'),
('1164622577'),
('1801988654'),
('1821404005'),
('1740273374'),
('1831295138'),
('1003887803'),
('1730167230'),
('1154324739'),
('1689765935'),
('1598773939'),
('1053726240'),
('1144282377'),
('1982660635'),
('1790993228'),
('1033129655'),
('1316149636'),
('1669451530'),
('1568447514'),
('1790999092'),
('1720055163'),
('1689766867'),
('1811190408'),
('1053497180'),
('1306953468'),
('1104260298'),
('1790107217'),
('1457591810'),
('1558490912'),
('1336336379'),
('1831348952'),
('1376509125'),
('1316130735'),
('1598098329'),
('1750375663'),
('1255862835'),
('1588858138'),
('1184816068'),
('1366530107'),
('1134206709'),
('1598963902'),
('1598723611'),
('1073136057'),
('1699213496'),
('1508870874'),
('1588850374'),
('1285704635'),
('1669678777'),
('1124073309'),
('1336229376'),
('1780947655'),
('1427497783'),
('1902018591'),
('1134184120'),
('1588920748'),
('1700901337'),
('1750377651'),
('1689992521'),
('1962471045'),
('1255599841'),
('1265485692'),
('1013206036'),
('1669649927'),
('1003062530'),
('1851658470'),
('1326289257'),
('1497839120'),
('1851370019'),
('1124003975'),
('1215092382'),
('1326276296'),
('1366703266'),
('1528243359'),
('1801984125'),
('1225053093'),
('1689644650'),
('1407112170'),
('1427340157'),
('1699230664'),
('1972874980'),
('1386731776'),
('1245482470'),
('1154973675'),
('1073670907'),
('1184920423'),
('1437391901'),
('1326207986'),
('1013013317'),
('1932423225'),
('1487006227'),
('1982876058'),
('1871626051'),
('1609160027'),
('1275357352'),
('1639277478'),
('1942265285'),
('1013241470'),
('1720271141'),
('1255385985'),
('1780898304'),
('1396712774'),
('1164454252'),
('1891952545'),
('1336489558'),
('1326337288'),
('1780856997'),
('1790700797'),
('1023541158'),
('1538128210'),
('1720178387'),
('1215114962'),
('1245281880'),
('1699859652'),
('1982794103'),
('1598970436'),
('1679747778'),
('1912999848'),
('1194941112'),
('1790963924'),
('1235319500'),
('1164456216'),
('1598292419'),
('1396769840'),
('1235155433'),
('1336325810'),
('1568543932'),
('1851711121'),
('1598785438'),
('1851317176'),
('1366642860'),
('1063614659'),
('1306007232'),
('1740716125'),
('1265621965'),
('1427051168'),
('1689646796'),
('1073577938'),
('1851310213'),
('1871907121'),
('1497996649'),
('1295052827'),
('1386689354'),
('1811968134'),
('1841204393'),
('1275625683'),
('1568482735'),
('1053471482'),
('1770803538'),
('1740247436'),
('1114455276'),
('1134141039'),
('1700140621'),
('1790913036'),
('1326171257'),
('1114134301'),
('1568588309'),
('1730195983'),
('1528273141'),
('1609063957'),
('1538397450'),
('1023305018'),
('1548367899'),
('1063701274'),
('1255646915'),
('1477585123'),
('1114022076'),
('1306116389'),
('1760448591'),
('1821168279'),
('1255351854'),
('1417918046'),
('1588819379'),
('1760602643'),
('1861433559'),
('1457408338'),
('1083928931'),
('1336180355'),
('1689692865'),
('1144271925'),
('1053738625'),
('1265879266'),
('1013175173'),
('1932310224'),
('1275643645'),
('1386702587'),
('1922022656'),
('1437162138'),
('1275746604'),
('1861449159'),
('1871561241'),
('1851684542'),
('1073752382'),
('1326410481'),
('1477529683'),
('1154595346'),
('1295826006'),
('1588995179'),
('1073728002'),
('1558574491'),
('1265460141'),
('1861444366'),
('1861689705'),
('1366605438'),
('1427449305'),
('1669657748'),
('1194097303'),
('1346313590'),
('1992052245'),
('1376683300'),
('1174547087'),
('1558330332'),
('1871529842'),
('1881950301'),
('1669573028'),
('1689130601'),
('1790047041'),
('1972760056'),
('1629041744'),
('1639186075'),
('1700061470'),
('1023360914'),
('1669669412'),
('1235366337'),
('1427475987'),
('1528268950'),
('1629033253'),
('1548362361'),
('1821288705'),
('1417107285'),
('1023097714'),
('1851614473'),
('1003882374'),
('1326034422'),
('1104881499'),
('1497770598'),
('1700189552'),
('1922198605'),
('1346268729'),
('1093895476'),
('1669752887'),
('1942599154'),
('1538189154'),
('1417094772'),
('1891024808'),
('1356319867'),
('1487880795'),
('1619139284'),
('1669408282'),
('1629239744'),
('1952620734'),
('1447300470'),
('1114292489'),
('1851391577'),
('1073946950'),
('1972729499'),
('1598050486'),
('1023048972'),
('1326275587'),
('1922018548'),
('1477840437'),
('1710920566'),
('1013994730'),
('1710940705'),
('1649389404'),
('1780651893'),
('1477644219'),
('1821172560'),
('1366098808'),
('1689854838'),
('1144266081'),
('1306196266'),
('1770920431'),
('1124213178'),
('1134156201'),
('1467507244'),
('1063463875'),
('1548269822'),
('1790005353'),
('1740336346'),
('1023060456'),
('1083820518'),
('1790719235'),
('1235124322'),
('1073678835'),
('1134113244'),
('1700871647'),
('1457428500'),
('1093138141'),
('1568445104'),
('1669459509'),
('1750947115'),
('1578934014'),
('1386719185'),
('1346534492'),
('1235106246'),
('1801942156'),
('1912305434'),
('1851369516'),
('1770774341'),
('1326369851'),
('1285763227'),
('1003802810'),
('1407837586'),
('1780725663'),
('1083700314'),
('1194868182'),
('1043305204'),
('1366799025'),
('1447264932'),
('1740418912'),
('1477757086'),
('1497060693'),
('1932491404'),
('1427377589'),
('1649368168'),
('1215956636'),
('1467689547'),
('1538463344'),
('1104864263'),
('1659565745'),
('1669964029'),
('1780774281'),
('1659507259'),
('1518027614'),
('1225378334'),
('1104032648'),
('1053574988'),
('1366478638'),
('1487659314'),
('1437204021'),
('1043265077'),
('1194013623'),
('1871907642'),
('1235559105'),
('1487645693'),
('1952412454'),
('1205912540'),
('1235367186'),
('1609192699'),
('1821409822'),
('1871889295'),
('1073685376'),
('1679500565'),
('1811925167'),
('1053430850'),
('1538249974'),
('1356480966'),
('1972693596'),
('1942456165'),
('1386977288'),
('1639596679'),
('1649200817'),
('1962436964'),
('1588863823'),
('1790841641'),
('1134474265'),
('1710952015'),
('1922063288'),
('1659312759'),
('1366049132'),
('1639458961'),
('1124117858'),
('1083168066'),
('1700810850'),
('1992997977'),
('1306072707'),
('1952544272'),
('1134193642'),
('1275557191'),
('1801292313'),
('1891704771'),
('1104916121'),
('1841361607'),
('1598847808'),
('1659532117'),
('1184670226'),
('1619119187'),
('1083737878'),
('1124034053'),
('1407114010'),
('1013189729'),
('1154519817'),
('1750657748'),
('1790809168'),
('1013951540'),
('1033168190'),
('1437142478'),
('1033553482'),
('1487836995'),
('1972988673'),
('1295834000'),
('1588666911'),
('1861415739'),
('1124203922'),
('1215018023'),
('1982699260'),
('1124017926'),
('1649474487'),
('1023273141'),
('1023144375'),
('1265676274'),
('1841332632'),
('1497734677'),
('1801969894'),
('1790713352'),
('1851357610'),
('1518028430'),
('1841255403'),
('1477717932'),
('1053561860'),
('1518276872'),
('1740297779'),
('1821018623'),
('1326113929'),
('1811098965'),
('1407210081'),
('1629075635'),
('1194046581'),
('1275503047'),
('1275743072'),
('1295794923'),
('1992767396'),
('1649716424'),
('1952428567'),
('1407219298'),
('1073923181'),
('1336294107'),
('1760438147'),
('1003932963'),
('1184957599'),
('1508887852'),
('1538508148'),
('1689777476'),
('1124213301'),
('1619961117'),
('1558528133'),
('1104807239'),
('1912172891'),
('1750474029'),
('1972857159'),
('1699863811'),
('1215146436'),
('1083761407'),
('1578552121'),
('1922049022'),
('1417248428'),
('1487102661'),
('1518071398'),
('1881731909'),
('1316985575'),
('1487840443'),
('1285687673'),
('1235312885'),
('1750588836'),
('1972526457'),
('1720437205'),
('1962592998'),
('1184675605'),
('1215072855'),
('1174739619'),
('1033224944'),
('1144336645'),
('1386734317'),
('1477670727'),
('1568442242'),
('1104980754'),
('1407936818'),
('1760632202'),
('1528120680'),
('1619134988'),
('1669902185'),
('1770663064'),
('1952324998'),
('1194794792'),
('1114933272'),
('1659815207'),
('1811157308'),
('1871850230'),
('1972997807'),
('1154321131'),
('1477649531'),
('1508132507'),
('1114341310'),
('1437400082'),
('1609018084'),
('1720200876'),
('1225010044'),
('1043272958'),
('1467650911'),
('1831258409'),
('1164445177'),
('1245320019'),
('1457409377'),
('1376934901'),
('1386652030'),
('1518055052'),
('1871611962'),
('1558550053'),
('1417994781'),
('1639224389'),
('1659661692'),
('1790214955'),
('1508059098'),
('1750744991'),
('1154526523'),
('1164678959'),
('1417063389'),
('1649347147'),
('1114587342'),
('1316039647'),
('1326073131'),
('1841413077'),
('1124331426'),
('1477681567'),
('1043563588'),
('1053361204'),
('1891927216'),
('1477651842'),
('1740441815'),
('1487745758'),
('1609857010'),
('1801846019'),
('1952409138'),
('1568481414'),
('1134155344'),
('1184917353'),
('1871589119'),
('1942275888'),
('1386969715'),
('1396061461'),
('1972691756'),
('1790977221'),
('1487751947'),
('1750301073'),
('1275596405'),
('1326068131'),
('1336160456'),
('1891823407'),
('1013192426'),
('1255324422'),
('1649297359'),
('1003254574'),
('1841242989'),
('1265407902'),
('1508132457'),
('1386907491'),
('1831156066'),
('1376852905'),
('1376581702'),
('1801897590'),
('1598023509'),
('1194959213'),
('1255499646'),
('1720166572'),
('1861802415'),
('1134544604'),
('1447315791'),
('1295791812'),
('1760641088'),
('1245253541'),
('1366715351'),
('1720304439'),
('1053360776'),
('1164413027'),
('1205935632'),
('1356399695'),
('1003134677'),
('1134534175'),
('1427245018'),
('1962527770'),
('1356384374'),
('1033512967'),
('1184620205'),
('1215154943'),
('1598059735'),
('1346367315'),
('1801116496'),
('1063690311'),
('1891889754'),
('1245349042'),
('1922011162'),
('1649723339'),
('1417984899'),
('1891734430'),
('1194450940'),
('1346279981'),
('1497118731'),
('1174970149'),
('1598103392'),
('1316186323'),
('1578707410'),
('1740254473'),
('1588197164'),
('1770603722'),
('1134102882'),
('1568553055'),
('1750487401'),
('1699260018'),
('1013128693'),
('1215324512'),
('1861658718'),
('1639432628'),
('1740244078'),
('1548425622'),
('1376650754'),
('1730584327'),
('1942615224'),
('1780843029'),
('1356553457'),
('1912509829'),
('1851607394'),
('1629150503'),
('1780034900'),
('1851757389'),
('1235112160'),
('1215424106'),
('1528080199'),
('1689931354'),
('1013956267'),
('1386690360'),
('1760634802'),
('1043871916'),
('1932257441'),
('1275733537'),
('1003221342'),
('1477516102'),
('1710199161'),
('1134270192'),
('1205315751'),
('1073613386'),
('1528049483'),
('1023450004'),
('1396934972'),
('1316451305'),
('1730783754'),
('1275624298'),
('1447646708'),
('1407976756'),
('1831308576'),
('1912071663'),
('1134360746'),
('1265630545'),
('1477573988'),
('1659874980'),
('1750590790'),
('1639185077'),
('1194932061'),
('1528055753'),
('1366746893'),
('1114173424'),
('1700841566'),
('1164628574'),
('1992872717'),
('1831285402'),
('1780073742'),
('1316056641'),
('1013018035'),
('1114306537'),
('1457779514'),
('1619026135'),
('1538106828'),
('1457580607'),
('1770903882'),
('1376025221'),
('1710977822'),
('1215464037'),
('1821435009'),
('1124081799'),
('1043226368'),
('1558395665'),
('1629184460'),
('1871680199'),
('1871649129'),
('1801886478'),
('1619298080'),
('1205852480'),
('1700839909'),
('1851711667'),
('1437248309'),
('1376591131'),
('1669779708'),
('1437339694'),
('1497980031'),
('1700869997'),
('1508829938'),
('1538480603'),
('1467428979'),
('1972600179'),
('1790191831'),
('1619001872'),
('1407974116'),
('1407143217'),
('1386963999'),
('1376832592'),
('1013934058'),
('1316904550'),
('1922623214'),
('1306885876'),
('1588033161'),
('1245240639'),
('1588602270'),
('1891112108'),
('1083036669'),
('1770749418'),
('1417915422'),
('1093982357'),
('1609343425'),
('1558484931'),
('1821080979'),
('1114944063'),
('1104820802'),
('1124257407'),
('1073873634'),
('1124292537'),
('1023081395'),
('1134788433'),
('1265788228'),
('1134362387'),
('1487116711'),
('1295768521'),
('1912056078'),
('1386148815'),
('1598840068'),
('1164008090'),
('1043300932'),
('1760846315'),
('1225068307'),
('1508116773'),
('1881097012'),
('1508892738'),
('1619242807'),
('1053491316'),
('1306805759'),
('1831402668'),
('1851599963'),
('1548226616'),
('1992961502'),
('1316946288'),
('1720230725'),
('1396721049'),
('1710901848'),
('1831167238'),
('1952863078'),
('1033149299'),
('1740790179'),
('1144286592'),
('1902875172'),
('1811275944'),
('1669630877'),
('1720054471'),
('1669448734'),
('1669811139'),
('1134118706'),
('1629399035'),
('1154384360'),
('1861876096'),
('1831137876'),
('1962608059'),
('1144481896'),
('1407947047'),
('1427562909'),
('1417295148'),
('1922251586'),
('1265572499'),
('1801472683'),
('1750519385'),
('1164963468'),
('1528047057'),
('1366695439'),
('1194990549'),
('1053951442'),
('1548578560'),
('1447335351'),
('1295368488'),
('1982995395'),
('1952584948'),
('1326332438'),
('1922172600'),
('1073565867'),
('1659378628'),
('1194734640'),
('1639345598'),
('1093193641'),
('1174575484'),
('1780942649'),
('1568444875'),
('1972519122'),
('1083969612'),
('1528084357'),
('1871834325'),
('1063916039'),
('1407852528'),
('1710958350'),
('1437389822'),
('1750647798'),
('1689787384'),
('1063451763'),
('1396039798'),
('1629079215'),
('1033130885'),
('1871591578'),
('1760846281'),
('1598072589'),
('1710245055'),
('1447584222'),
('1558335083'),
('1295749422'),
('1518599646'),
('1588641740'),
('1649208521'),
('1912909615'),
('1508245572'),
('1285697490'),
('1750551420'),
('1386873677'),
('1942430715'),
('1689948150'),
('1619185675'),
('1356337273'),
('1780641555'),
('1780618462'),
('1992756373'),
('1336194125'),
('1083870919'),
('1821268293'),
('1346284957'),
('1700835832'),
('1063556991'),
('1639326275'),
('1487816443'),
('1902986243'),
('1508011776'),
('1205002797'),
('1154725844'),
('1497000566'),
('1982809315'),
('1326178302'),
('1811464621'),
('1033177928'),
('1295940336'),
('1093776585'),
('1780669390'),
('1346298684'),
('1265634562'),
('1437123247'),
('1154339570'),
('1356444749'),
('1881617835'),
('1316901549'),
('1609119015'),
('1336350891'),
('1861417271'),
('1801184973'),
('1063505030'),
('1013954619'),
('1235565409'),
('1578924247'),
('1124013826'),
('1629363163'),
('1871703678'),
('1386955201'),
('1952515538'),
('1366732893'),
('1801060462'),
('1255534855'),
('1093776668'),
('1669890745'),
('1033384805'),
('1609839844'),
('1932149499'),
('1376579870'),
('1770542359'),
('1114240561'),
('1063407575'),
('1922172162'),
('1609854538'),
('1114319324'),
('1891792040'),
('1760491781'),
('1902163926'),
('1619963246'),
('1154536308'),
('1255687034'),
('1083895338'),
('1386890135'),
('1376646695'),
('1033475793'),
('1568648061'),
('1013194513'),
('1831499409'),
('1710971478'),
('1609099431'),
('1821016502'),
('1710936448'),
('1669423562'),
('1821038548'),
('1336240332'),
('1124342662'),
('1528453313'),
('1821165762'),
('1790705861'),
('1326363318'),
('1356379556'),
('1962413492'),
('1992037931'),
('1275617375'),
('1962744797'),
('1386664019'),
('1003903493'),
('1265962559'),
('1255436697'),
('1598923096'),
('1053514844'),
('1568492940'),
('1144230962'),
('1669636536'),
('1558492199'),
('1104881697'),
('1194843722'),
('1821282484'),
('1831489053'),
('1376553255'),
('1073763140'),
('1831416882'),
('1740424621'),
('1548436447'),
('1245399807'),
('1114992203'),
('1295907194'),
('1801963749'),
('1003080078'),
('1275732372'),
('1851548960'),
('1871895573'),
('1346623840'),
('1487621199'),
('1194046318'),
('1992890883'),
('1407860331'),
('1619135845'),
('1760640403'),
('1699861047'),
('1952422719'),
('1326144486'),
('1316172703'),
('1518932250'),
('1467612382'),
('1841630142'),
('1861694887'),
('1700102597'),
('1346425584'),
('1669537478'),
('1902833684'),
('1730220666'),
('1326210311'),
('1891885554'),
('1235492547'),
('1891945895'),
('1104913045'),
('1255415360'),
('1407108913'),
('1184633059'),
('1760485890'),
('1013993740'),
('1083698021'),
('1154409423'),
('1265494835'),
('1851357388'),
('1114976214'),
('1013119320'),
('1518050608'),
('1396906368'),
('1982654448'),
('1154318970'),
('1609162387'),
('1609219989'),
('1467537415'),
('1790757557'),
('1760571467'),
('1619066149'),
('1508007675'),
('1093815904'),
('1962418145'),
('1811961162'),
('1396787602'),
('1093809659'),
('1760707046'),
('1548292402'),
('1477796068'),
('1336140441'),
('1699917104'),
('1043224140'),
('1912112020'),
('1194906321'),
('1831131606'),
('1699934091'),
('1669682837'),
('1306815014'),
('1245527787'),
('1831109461'),
('1427076454'),
('1467512871'),
('1093741175'),
('1629426085'),
('1548211642'),
('1649251422'),
('1588778955'),
('1114188422'),
('1366538811'),
('1932555109'),
('1659344463'),
('1255480042'),
('1881868511'),
('1992931182'),
('1851341119'),
('1326091562'),
('1447281969'),
('1396766093'),
('1689764201'),
('1831223049'),
('1295793172'),
('1407828411'),
('1508055468'),
('1740246743'),
('1578705612'),
('1679951347'),
('1932423530'),
('1477697068'),
('1407177124'),
('1467465971'),
('1154529774'),
('1477744258'),
('1245321595'),
('1477701654'),
('1538185756'),
('1962442079'),
('1366426140'),
('1033313978'),
('1336179662'),
('1760642656'),
('1821248451'),
('1386662062')
),

hcp_engaged AS (
  SELECT DISTINCT
    TRY_CAST(b.npi__v AS STRING) AS hcp_npi
  FROM com_edp_prd.com_raw.vcrm_call2__v AS a
  LEFT JOIN com_intgr.customer AS b
    ON a.account__v = b.id
  INNER JOIN input_hcps i
    ON TRY_CAST(b.npi__v AS STRING) = TRY_CAST(i.hcp_npi AS STRING)
  WHERE a.call_date__v BETWEEN '2025-07-01' AND '2025-12-31'
    AND b.npi__v IS NOT NULL
),

hcp_profiled AS (
  SELECT DISTINCT
    TRY_CAST(b.npi__v AS STRING) AS hcp_npi
  FROM com_intgr.survey_target AS a
  LEFT JOIN com_intgr.customer AS b
    ON a.account__v = b.id
  INNER JOIN input_hcps i
    ON TRY_CAST(b.npi__v AS STRING) = TRY_CAST(i.hcp_npi AS STRING)
  WHERE b.npi__v IS NOT NULL
),

unique_patients_by_hcp AS (
  -- collapse to 1 row per HCP (your all_patients_unique_hcp can have multiple rows per HCP by specialty)
  SELECT
    TRY_CAST(hcp_npi AS STRING) AS hcp_npi,
    MAX(all_patients_unique) AS all_patients_unique
  FROM all_patients_unique_hcp
  GROUP BY 1
)

SELECT
  TRY_CAST(i.hcp_npi AS STRING) AS hcp_npi,

  CASE WHEN e.hcp_npi IS NOT NULL THEN 1 ELSE 0 END AS is_hcp_engaged,
  CASE WHEN p.hcp_npi IS NOT NULL THEN 1 ELSE 0 END AS is_hcp_profiled,

  COALESCE(nu.all_patients_non_unique, 0) AS all_patients_non_unique,
  COALESCE(u.all_patients_unique, 0)      AS all_patients_unique

FROM input_hcps i
LEFT JOIN hcp_engaged e
  ON TRY_CAST(i.hcp_npi AS STRING) = e.hcp_npi
LEFT JOIN hcp_profiled p
  ON TRY_CAST(i.hcp_npi AS STRING) = p.hcp_npi
LEFT JOIN all_patients_non_unique_hcp nu
  ON TRY_CAST(i.hcp_npi AS STRING) = TRY_CAST(nu.hcp_npi AS STRING)
LEFT JOIN unique_patients_by_hcp u
  ON TRY_CAST(i.hcp_npi AS STRING) = u.hcp_npi
;
